In [ ]:
# Нормализация - ОБЯЗАТЕЛЬНО для разных масштабов признаков

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[features])

In [ ]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        
        # Энкодер - сжимает последовательность
        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        
        # Декодер - восстанавливает последовательность
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=input_size,
            num_layers=num_layers,
            batch_first=True
        )
    
    def forward(self, x):
        # Сжимаем
        encoded, (hidden, cell) = self.encoder(x)
        # Восстанавливаем
        decoded, _ = self.decoder(encoded)
        return decoded

# Создание скользящих окон (окно = 60 точек = 1 минута если 1Hz)
def create_windows(data, window_size=60):
    windows = []
    for i in range(len(data) - window_size):
        windows.append(data[i:i+window_size])
    return np.array(windows)

window_size = 60
X = create_windows(df_scaled, window_size)
X_tensor = torch.FloatTensor(X)

# Обучение
model = LSTMAutoencoder(input_size=20, hidden_size=32, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(50):
    optimizer.zero_grad()
    output = model(X_tensor)
    loss = criterion(output, X_tensor)
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

# Ошибка восстановления = оценка аномальности
with torch.no_grad():
    reconstructed = model(X_tensor)
    # Чем больше ошибка - тем аномальнее
    reconstruction_error = torch.mean(
        (X_tensor - reconstructed) ** 2, 
        dim=[1, 2]
    ).numpy()

# Порог аномалии
threshold = np.percentile(reconstruction_error, 95)
anomalies = reconstruction_error > threshold

### Визуализация аномалий

##### 1. Временной ряд с подсветкой аномалий

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(4, 1, figsize=(15, 12))

# Выбираем ключевые признаки для отображения
key_features = ['altitude', 'speed', 'pitch', 'roll']

for idx, feature in enumerate(key_features):
    ax = axes[idx]
    
    # Нормальные точки
    ax.plot(df.index, df[feature], 
            color='steelblue', linewidth=0.8, label='Норма')
    
    # Подсветка аномальных зон
    anomaly_mask = df['anomaly'] == -1
    ax.fill_between(
        df.index, 
        df[feature].min(), 
        df[feature].max(),
        where=anomaly_mask,
        color='red', 
        alpha=0.3, 
        label='Аномалия'
    )
    
    # Точки аномалий
    ax.scatter(
        df.index[anomaly_mask], 
        df[feature][anomaly_mask],
        color='red', s=20, zorder=5
    )
    
    ax.set_ylabel(feature)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Детекция аномалий полёта', fontsize=14)
plt.tight_layout()
plt.show()

#### 2. Тепловая карта аномальности по признакам

In [ ]:
import seaborn as sns

# Считаем вклад каждого признака в аномалию
fig, ax = plt.subplots(figsize=(15, 6))

# Нормализованные значения во времени
sns.heatmap(
    df_scaled.T,  # признаки × время
    cmap='RdYlGn_r',
    ax=ax,
    yticklabels=features
)

# Добавляем маркеры аномалий
anomaly_times = df.index[df['anomaly'] == -1]
for t in anomaly_times:
    ax.axvline(x=t, color='red', alpha=0.5, linewidth=0.5)

ax.set_title('Тепловая карта признаков (красные линии = аномалии)')
plt.show()

#### 3. График ошибки восстановления (для Autoencoder)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

# Ошибка восстановления
ax1.plot(reconstruction_error, color='steelblue', linewidth=0.8)
ax1.axhline(y=threshold, color='red', linestyle='--', 
            label=f'Порог = {threshold:.3f}')
ax1.fill_between(
    range(len(reconstruction_error)),
    reconstruction_error,
    threshold,
    where=reconstruction_error > threshold,
    color='red', alpha=0.4, label='Аномалия'
)
ax1.set_ylabel('Ошибка восстановления')
ax1.legend()
ax1.set_title('LSTM Autoencoder: ошибка восстановления')

# Высота для контекста
ax2.plot(df['altitude'].values[window_size:], color='green')
ax2.set_ylabel('Высота')
ax2.set_xlabel('Время')

plt.tight_layout()
plt.show()

#### 4. PCA визуализация в 2D

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
df_2d = pca.fit_transform(df_scaled)

plt.figure(figsize=(10, 8))

# Нормальные точки
normal_mask = df['anomaly'] == 1
plt.scatter(df_2d[normal_mask, 0], df_2d[normal_mask, 1],
            c='steelblue', alpha=0.5, s=10, label='Норма')

# Аномалии
anomaly_mask = df['anomaly'] == -1
plt.scatter(df_2d[anomaly_mask, 0], df_2d[anomaly_mask, 1],
            c='red', alpha=0.8, s=50, label='Аномалия', marker='x')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} дисперсии)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} дисперсии)')
plt.title('PCA проекция: нормальные vs аномальные точки')
plt.legend()
plt.show()